In [1]:
# =========================================================
# RPDA8412 RESEARCH PROJECT
# NOTEBOOK 04: VOLATILITY FEATURE ENGINEERING
# =========================================================

from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FINAL_DIR = PROJECT_ROOT / "data" / "final"

df = pd.read_csv(
    FINAL_DIR / "daily_forecast_base_provisional.csv",
    parse_dates=["Date"]
)

df = df.sort_values("Date").reset_index(drop=True)

print("=" * 60)
print("VOLATILITY FEATURE ENGINEERING")
print("=" * 60)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

print("\nMissing values:")
print(df.isna().sum())

display(df.head())

VOLATILITY FEATURE ENGINEERING
Dataset shape: (4024, 9)
Date range: 2010-01-04 00:00:00 to 2025-12-31 00:00:00

Missing values:
Date         0
Adj_Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
GPR_lag1     0
EPU_lag1     0
dtype: int64


,Date,Adj_Close,Close,High,Low,Open,Volume,GPR_lag1,EPU_lag1
0,2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000,91.471298,216.95
1,2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,177.117065,150.35
2,2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,122.525620,151.16
3,2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,161.588669,95.95
4,2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,102.262291,92.20


In [2]:
# =========================================================
# STEP 1: CALCULATE DAILY LOG RETURNS
# =========================================================

df["Log_Return"] = np.log(
    df["Close"] / df["Close"].shift(1)
)

print("=" * 60)
print("DAILY LOG RETURNS")
print("=" * 60)

print("\nDescriptive statistics:")
display(df["Log_Return"].describe())

print("\nMissing values:")
print(df["Log_Return"].isna().sum())

print("\nFirst 10 observations:")
display(
    df[["Date", "Close", "Log_Return"]].head(10)
)

DAILY LOG RETURNS

Descriptive statistics:


count    4023.000000
mean        0.000447
std         0.010942
min        -0.127652
25%        -0.003811
50%         0.000702
75%         0.005675
max         0.090895
Name: Log_Return, dtype: float64


Missing values:
1

First 10 observations:


,Date,Close,Log_Return
0,2010-01-04,1132.989990,NaN
1,2010-01-05,1136.520020,0.003111
2,2010-01-06,1137.140015,0.000545
3,2010-01-07,1141.689941,0.003993
4,2010-01-08,1144.979980,0.002878
5,2010-01-11,1146.979980,0.001745
6,2010-01-12,1136.219971,-0.009425
7,2010-01-13,1145.680054,0.008291
8,2010-01-14,1148.459961,0.002423
9,2010-01-15,1136.030029,-0.010882


In [3]:
# =========================================================
# STEP 2: HISTORICAL REALISED VOLATILITY
# =========================================================

df["Squared_Return"] = df["Log_Return"] ** 2

df["RV_Historical_20D"] = np.sqrt(
    df["Squared_Return"].rolling(
        window=20,
        min_periods=20
    ).sum()
)

print("=" * 60)
print("HISTORICAL REALISED VOLATILITY")
print("=" * 60)

print("\nDescriptive statistics:")
display(df["RV_Historical_20D"].describe())

print("\nMissing values:")
print(df["RV_Historical_20D"].isna().sum())

display(
    df[
        [
            "Date",
            "Log_Return",
            "RV_Historical_20D"
        ]
    ].head(25)
)

HISTORICAL REALISED VOLATILITY

Descriptive statistics:


count    4004.000000
mean        0.041530
std         0.026071
min         0.009526
25%         0.026326
50%         0.035147
75%         0.049002
max         0.270908
Name: RV_Historical_20D, dtype: float64


Missing values:
20


,Date,Log_Return,RV_Historical_20D
0,2010-01-04,NaN,NaN
1,2010-01-05,0.003111,NaN
2,2010-01-06,0.000545,NaN
3,2010-01-07,0.003993,NaN
4,2010-01-08,0.002878,NaN
5,2010-01-11,0.001745,NaN
6,2010-01-12,-0.009425,NaN
7,2010-01-13,0.008291,NaN
8,2010-01-14,0.002423,NaN
9,2010-01-15,-0.010882,NaN


In [4]:
# =========================================================
# STEP 3: FUTURE REALISED VOLATILITY TARGETS
# =========================================================

def calculate_future_rv(squared_returns, horizon):
    """
    Calculate realised volatility over the next
    'horizon' trading days, excluding the current day.
    """

    future_squared = sum(
        squared_returns.shift(-i)
        for i in range(1, horizon + 1)
    )

    # Require every future return to be available
    future_count = sum(
        squared_returns.shift(-i).notna().astype(int)
        for i in range(1, horizon + 1)
    )

    return np.sqrt(
        future_squared.where(future_count == horizon)
    )


df["RV_Future_1D"] = calculate_future_rv(
    df["Squared_Return"], 1
)

df["RV_Future_5D"] = calculate_future_rv(
    df["Squared_Return"], 5
)

df["RV_Future_20D"] = calculate_future_rv(
    df["Squared_Return"], 20
)

print("=" * 60)
print("FUTURE VOLATILITY TARGETS")
print("=" * 60)

print("\nMissing values:")
print(
    df[
        [
            "RV_Future_1D",
            "RV_Future_5D",
            "RV_Future_20D"
        ]
    ].isna().sum()
)

print("\nDescriptive statistics:")
display(
    df[
        [
            "RV_Future_1D",
            "RV_Future_5D",
            "RV_Future_20D"
        ]
    ].describe()
)

display(
    df[
        [
            "Date",
            "RV_Historical_20D",
            "RV_Future_1D",
            "RV_Future_5D",
            "RV_Future_20D"
        ]
    ].head(25)
)

FUTURE VOLATILITY TARGETS

Missing values:
RV_Future_1D      1
RV_Future_5D      5
RV_Future_20D    20
dtype: int64

Descriptive statistics:


,RV_Future_1D,RV_Future_5D,RV_Future_20D
count,4023.000000,4019.000000,4004.000000
mean,0.007225,0.019598,0.041530
std,0.008229,0.014697,0.026071
min,0.000000,0.001362,0.009526
25%,0.001973,0.010600,0.026326
50%,0.004837,0.016385,0.035147
75%,0.009742,0.024053,0.049002
max,0.127652,0.200982,0.270908


,Date,RV_Historical_20D,RV_Future_1D,RV_Future_5D,RV_Future_20D
0,2010-01-04,NaN,0.003111,0.006103,0.046068
1,2010-01-05,NaN,0.000545,0.010789,0.046290
2,2010-01-06,NaN,0.003993,0.013596,0.056065
3,2010-01-07,NaN,0.002878,0.013221,0.055997
4,2010-01-08,NaN,0.001745,0.016880,0.056627
5,2010-01-11,NaN,0.009425,0.020885,0.058064
6,2010-01-12,NaN,0.008291,0.021468,0.057338
7,2010-01-13,NaN,0.002423,0.027531,0.057547
8,2010-01-14,NaN,0.010882,0.035403,0.057562
9,2010-01-15,NaN,0.012422,0.034000,0.059273


In [5]:
# =========================================================
# STEP 4: VOLATILITY VALIDATION
# =========================================================

print("=" * 60)
print("VOLATILITY VALIDATION")
print("=" * 60)

assert df["Log_Return"].isna().sum() == 1

assert df["RV_Historical_20D"].isna().sum() == 20

assert df["RV_Future_1D"].isna().sum() == 1

assert df["RV_Future_5D"].isna().sum() == 5

assert df["RV_Future_20D"].isna().sum() == 20

# Confirm one-day target equals next-day absolute return
expected_1d = df["Log_Return"].shift(-1).abs()

assert np.allclose(
    df["RV_Future_1D"].dropna(),
    expected_1d.dropna()
)

# Manually verify one 5-day target
manual_5d = np.sqrt(
    df.loc[1:5, "Squared_Return"].sum()
)

calculated_5d = df.loc[0, "RV_Future_5D"]

assert np.isclose(
    manual_5d,
    calculated_5d
)

# Check nonnegative volatility
volatility_columns = [
    "RV_Historical_20D",
    "RV_Future_1D",
    "RV_Future_5D",
    "RV_Future_20D"
]

for column in volatility_columns:
    assert (
        df[column].dropna() >= 0
    ).all(), f"Negative volatility detected: {column}"

print("\nALL VOLATILITY VALIDATION CHECKS PASSED!")

VOLATILITY VALIDATION

ALL VOLATILITY VALIDATION CHECKS PASSED!


In [6]:
# =========================================================
# STEP 5: SAVE ENGINEERED DATASET
# =========================================================

output_file = (
    FINAL_DIR / "daily_volatility_engineered_provisional.csv"
)

df.to_csv(
    output_file,
    index=False
)

print("Volatility dataset saved successfully!")
print("File:", output_file)
print("Observations:", len(df))
print("Columns:", len(df.columns))

Volatility dataset saved successfully!
File: C:\Users\ibrah\RPDA8412 Research Project\data\final\daily_volatility_engineered_provisional.csv
Observations: 4024
Columns: 15
